# TP3 Chatbots basados en recuperación de la información

En inglés information retrieval chatbots

# Motor de búsqueda

* Búsqueda por palabras clave: Extrae palabras clave de la pregunta del usuario y busca coincidencias en las preguntas almacenadas.

* Similitud del coseno: Si has representado las preguntas como vectores (por ejemplo, usando TF-IDF o word embeddings), puedes usar la similitud del coseno para medir la distancia entre las preguntas.

* Embeddings: Utiliza modelos de word embeddings como por ejemplo Word2Vec para obtener representaciones semánticas de las preguntas y las consultas del usuario.

## Librerías

Debes trabajar en Python. Puedes usar las librerías sklearn, pandas, spacy o nltk o gensim para el punto de usar o buscar embeddings.

In [20]:
!pip install spacy --quiet
!python -m spacy download es_core_news_sm --quiet
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')


## Actividades



### 1) Elaborar un dataset de preguntas y respuestas para crear un Chatbot para un aplicación particular. ( 3 puntos )

1.1 Debe definir la aplicación (atención al cliente bancario, atención a estudiantes universitarios, etc).

1.2 El listado de preguntas y respuestas debe tener como mínimo 20 elementos pregunta - respuesta.

###  2) Crear el chatbot utilizando TFIDF y similitud del coseno. (1 punto)

### 3) Crear otro chatbot utilizando embeddings. Indique cuál embedding (1 punto) pre-entrenado eligió.

### 4) Muestra ambos chatbots funcionando (1 punto)

Adjuntar la lista de preguntas y respuestas utilizadas para probar el funcionamiento.

Releve o indique cuáles respondió correctamente y cuáles no.

### 5) Añade tus conclusiones de todo lo realizado (2 punto)

* Resalte e indique en cuáles respuestas falla o tiene problemas.
* Cuáles preguntas confunde.
* Compare los resultados de los chatbots.



### No olvides:

* Explicar tus decisiones y configuraciones. Añadir tus conclusiones.
* Anunciar en el foro cuál será tu aplicación y postear tu entrega y tus avances.
* Debes subir tu notebook a un repo GitHub público de tu propiedad compartido + enlace colab.
* Documentar todo el proceso.
* Citar tus fuentes





## 1. Definición de la aplicación

La aplicación que elegí es un chatbot de orientación para estudiantes que cursan a distancia o en modalidad asincrónica.  
El objetivo es que pueda responder dudas frecuentes sobre cursada, materias, entregas, parciales, clases grabadas, uso de Google Colab, GitHub y comunicación con docentes.

Elegí este caso porque es cercano a una situación real de estudio. Cuando la cursada es a distancia, muchas consultas se repiten y no siempre uno puede asistir a la clase en vivo. En esos casos, un chatbot de recuperación puede ayudar a encontrar rápido una respuesta ya cargada.

In [21]:
import re
import numpy as np
import pandas as pd
import spacy

## 2. Dataset de preguntas y respuestas

A continuación armé una base de conocimiento con más de 20 elementos de pregunta-respuesta.  
Cada fila representa una posible intención del usuario y una respuesta preparada.

En un chatbot de recuperación, esta base es muy importante: si las preguntas cargadas son pobres o incompletas, el chatbot va a tener más dificultades para encontrar una respuesta correcta.

In [22]:
qa_data = [
    {
        "id": "P01",
        "tema": "inscripcion",
        "pregunta": "¿Cómo me inscribo a una materia?",
        "respuesta": "Para inscribirte a una materia tenés que ingresar al sistema académico, buscar la materia disponible y confirmar la inscripción dentro del período habilitado."
    },
    {
        "id": "P02",
        "tema": "inscripcion",
        "pregunta": "¿Dónde veo las materias disponibles?",
        "respuesta": "Las materias disponibles se consultan en el sistema académico o en el aula virtual, según la información publicada por la institución."
    },
    {
        "id": "P03",
        "tema": "aula_virtual",
        "pregunta": "¿Cómo entro al aula virtual?",
        "respuesta": "Para entrar al aula virtual tenés que usar el usuario y contraseña que te dio la institución. Si no podés ingresar, conviene revisar el correo o pedir recuperación de clave."
    },
    {
        "id": "P04",
        "tema": "aula_virtual",
        "pregunta": "¿Qué hago si no puedo acceder al campus?",
        "respuesta": "Si no podés acceder al campus, primero verificá tu usuario, contraseña y conexión. Si el problema sigue, tenés que comunicarte con soporte o administración."
    },
    {
        "id": "P05",
        "tema": "trabajos_practicos",
        "pregunta": "¿Dónde se entregan los trabajos prácticos?",
        "respuesta": "Los trabajos prácticos normalmente se entregan en el aula virtual, dentro de la sección de la materia y antes de la fecha límite indicada por el docente."
    },
    {
        "id": "P06",
        "tema": "trabajos_practicos",
        "pregunta": "¿Puedo entregar un trabajo práctico fuera de término?",
        "respuesta": "La entrega fuera de término depende de la política de cada materia. Lo recomendable es consultar al docente y justificar el motivo de la demora."
    },
    {
        "id": "P07",
        "tema": "evaluaciones",
        "pregunta": "¿Dónde puedo ver las fechas de los parciales?",
        "respuesta": "Las fechas de parciales suelen publicarse en el aula virtual, en el cronograma de la materia o en los comunicados del docente."
    },
    {
        "id": "P08",
        "tema": "evaluaciones",
        "pregunta": "¿Qué pasa si desapruebo un parcial?",
        "respuesta": "Si desaprobás un parcial, generalmente podés acceder a una instancia de recuperatorio, aunque eso depende del reglamento de la materia."
    },
    {
        "id": "P09",
        "tema": "evaluaciones",
        "pregunta": "¿Cómo sé si aprobé una materia?",
        "respuesta": "Para saber si aprobaste una materia tenés que revisar tus notas, las condiciones de regularidad y la información final publicada por el docente o la institución."
    },
    {
        "id": "P10",
        "tema": "asistencia",
        "pregunta": "¿Es obligatoria la asistencia a clases?",
        "respuesta": "La asistencia puede depender de la modalidad y de la materia. Conviene revisar el programa o consultar al docente para saber el requisito exacto."
    },
    {
        "id": "P11",
        "tema": "asistencia",
        "pregunta": "¿Qué hago si falto a una clase?",
        "respuesta": "Si faltás a una clase, lo mejor es revisar el material subido al aula virtual, pedir apuntes a un compañero y consultar si hubo alguna actividad obligatoria."
    },
    {
        "id": "P12",
        "tema": "comunicacion",
        "pregunta": "¿Cómo me comunico con un docente?",
        "respuesta": "Podés comunicarte con un docente por el aula virtual, correo institucional o el medio que haya indicado al inicio de la materia."
    },
    {
        "id": "P13",
        "tema": "comunicacion",
        "pregunta": "¿Dónde se publican los avisos importantes?",
        "respuesta": "Los avisos importantes suelen publicarse en el aula virtual, por correo electrónico o en los canales oficiales de la institución."
    },
    {
        "id": "P14",
        "tema": "colab",
        "pregunta": "¿Para qué se usa Google Colab?",
        "respuesta": "Google Colab se usa para escribir y ejecutar notebooks de Python en la nube, sin tener que instalar todo el entorno en la computadora."
    },
    {
        "id": "P15",
        "tema": "colab",
        "pregunta": "¿Cómo comparto un notebook de Google Colab?",
        "respuesta": "Para compartir un notebook de Colab tenés que usar el botón Compartir, cambiar los permisos a cualquier persona con el enlace y copiar el link."
    },
    {
        "id": "P16",
        "tema": "github",
        "pregunta": "¿Para qué sirve GitHub en los trabajos prácticos?",
        "respuesta": "GitHub sirve para guardar el código, documentar el proyecto y compartir el repositorio de forma pública o privada según lo pedido por la consigna."
    },
    {
        "id": "P17",
        "tema": "github",
        "pregunta": "¿Qué archivo no debería faltar en un repositorio?",
        "respuesta": "Un archivo importante es el README, porque explica de qué trata el proyecto, cómo se ejecuta y qué contiene el repositorio."
    },
    {
        "id": "P18",
        "tema": "python",
        "pregunta": "¿Qué hago si Python me muestra un error?",
        "respuesta": "Cuando Python muestra un error, conviene leer el mensaje, identificar la línea donde ocurre y revisar si el problema es de sintaxis, datos o librerías."
    },
    {
        "id": "P19",
        "tema": "python",
        "pregunta": "¿Por qué tengo que comentar el código?",
        "respuesta": "Comentar el código ayuda a explicar qué hace cada parte y facilita que otra persona pueda entender el proceso realizado."
    },
    {
        "id": "P20",
        "tema": "chatbot",
        "pregunta": "¿Qué es un chatbot basado en recuperación de información?",
        "respuesta": "Es un chatbot que no inventa una respuesta nueva, sino que busca la pregunta más parecida en una base de conocimiento y devuelve la respuesta asociada."
    },
    {
        "id": "P21",
        "tema": "chatbot",
        "pregunta": "¿Cuál es la diferencia entre TF-IDF y embeddings?",
        "respuesta": "TF-IDF representa textos según la importancia de las palabras, mientras que los embeddings intentan capturar relaciones de significado entre palabras o frases."
    },
    {
        "id": "P22",
        "tema": "chatbot",
        "pregunta": "¿Qué significa similitud del coseno?",
        "respuesta": "La similitud del coseno mide qué tan parecidos son dos vectores según la dirección que tienen. Cuanto más cerca de 1, más parecidos son."
    },
    {
        "id": "P23",
        "tema": "regularidad",
        "pregunta": "¿Qué significa regularizar una materia?",
        "respuesta": "Regularizar una materia significa cumplir las condiciones mínimas de cursada, como asistencia, trabajos prácticos y evaluaciones, según el reglamento."
    },
    {
        "id": "P24",
        "tema": "certificados",
        "pregunta": "¿Dónde pido un certificado de alumno regular?",
        "respuesta": "El certificado de alumno regular se solicita en administración o por el medio institucional indicado para trámites académicos."
    },
    {
        "id": "P25",
        "tema": "organizacion",
        "pregunta": "¿Cómo puedo organizarme mejor para estudiar?",
        "respuesta": "Una forma simple de organizarte es revisar el cronograma, anotar fechas importantes y dividir los trabajos prácticos en pasos chicos."
    }
]

df_qa = pd.DataFrame(qa_data)
df_qa

,id,tema,pregunta,respuesta
0,P01,inscripcion,¿Cómo me inscribo a una materia?,Para inscribirte a una materia tenés que ingre...
1,P02,inscripcion,¿Dónde veo las materias disponibles?,Las materias disponibles se consultan en el si...
2,P03,aula_virtual,¿Cómo entro al aula virtual?,Para entrar al aula virtual tenés que usar el ...
3,P04,aula_virtual,¿Qué hago si no puedo acceder al campus?,"Si no podés acceder al campus, primero verific..."
4,P05,trabajos_practicos,¿Dónde se entregan los trabajos prácticos?,Los trabajos prácticos normalmente se entregan...
5,P06,trabajos_practicos,¿Puedo entregar un trabajo práctico fuera de t...,La entrega fuera de término depende de la polí...
6,P07,evaluaciones,¿Dónde puedo ver las fechas de los parciales?,Las fechas de parciales suelen publicarse en e...
7,P08,evaluaciones,¿Qué pasa si desapruebo un parcial?,"Si desaprobás un parcial, generalmente podés a..."
8,P09,evaluaciones,¿Cómo sé si aprobé una materia?,Para saber si aprobaste una materia tenés que ...
9,P10,asistencia,¿Es obligatoria la asistencia a clases?,La asistencia puede depender de la modalidad y...


In [23]:
print("Cantidad de preguntas y respuestas:", len(df_qa))
df_qa["tema"].value_counts()

Cantidad de preguntas y respuestas: 25


tema
evaluaciones          3
chatbot               3
inscripcion           2
aula_virtual          2
trabajos_practicos    2
asistencia            2
comunicacion          2
colab                 2
github                2
python                2
regularidad           1
certificados          1
organizacion          1
Name: count, dtype: int64

## 3. Preprocesamiento simple del texto

Antes de comparar las consultas, hice una limpieza básica:

- pasar todo a minúsculas.
- quitar espacios repetidos.
- quitar signos que no aportan demasiado para este caso.

No hice una limpieza demasiado agresiva porque algunas palabras cortas pueden seguir siendo útiles dentro de las preguntas.

In [24]:
def normalizar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r"[^a-záéíóúüñ0-9\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

df_qa["pregunta_normalizada"] = df_qa["pregunta"].apply(normalizar_texto)
df_qa[["id", "pregunta", "pregunta_normalizada"]].head()

,id,pregunta,pregunta_normalizada
0,P01,¿Cómo me inscribo a una materia?,cómo me inscribo a una materia
1,P02,¿Dónde veo las materias disponibles?,dónde veo las materias disponibles
2,P03,¿Cómo entro al aula virtual?,cómo entro al aula virtual
3,P04,¿Qué hago si no puedo acceder al campus?,qué hago si no puedo acceder al campus
4,P05,¿Dónde se entregan los trabajos prácticos?,dónde se entregan los trabajos prácticos


## 4. Chatbot 1: TF-IDF + similitud del coseno

TF-IDF transforma las preguntas en vectores numéricos.  
Después comparo la consulta del usuario con todas las preguntas del dataset usando similitud del coseno.

Configuraciones usadas:

- `ngram_range=(1,2)`: para considerar palabras sueltas y pares de palabras.
- `strip_accents="unicode"`: para reducir problemas con acentos.
- `min_df=1`: porque el dataset es chico y no conviene eliminar términos.

In [25]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=1
)

tfidf_matrix = vectorizer.fit_transform(df_qa["pregunta_normalizada"])

print("Tamaño de la matriz TF-IDF:", tfidf_matrix.shape)

Tamaño de la matriz TF-IDF: (25, 222)


In [26]:
def responder_tfidf(consulta, umbral=0.16):
    consulta_normalizada = normalizar_texto(consulta)
    consulta_vector = vectorizer.transform([consulta_normalizada])

    similitudes = cosine_similarity(consulta_vector, tfidf_matrix).flatten()
    indice_mejor = similitudes.argmax()
    mejor_similitud = similitudes[indice_mejor]

    if mejor_similitud < umbral:
        return {
            "modelo": "TF-IDF",
            "consulta": consulta,
            "id_recuperado": None,
            "pregunta_recuperada": None,
            "similitud": round(float(mejor_similitud), 4),
            "respuesta": "No encontré una respuesta segura para esa consulta. Probá reformular la pregunta."
        }

    fila = df_qa.iloc[indice_mejor]
    return {
        "modelo": "TF-IDF",
        "consulta": consulta,
        "id_recuperado": fila["id"],
        "pregunta_recuperada": fila["pregunta"],
        "similitud": round(float(mejor_similitud), 4),
        "respuesta": fila["respuesta"]
    }

responder_tfidf("¿Cómo subo mi notebook de Colab?")

{'modelo': 'TF-IDF',
 'consulta': '¿Cómo subo mi notebook de Colab?',
 'id_recuperado': 'P15',
 'pregunta_recuperada': '¿Cómo comparto un notebook de Google Colab?',
 'similitud': 0.5899,
 'respuesta': 'Para compartir un notebook de Colab tenés que usar el botón Compartir, cambiar los permisos a cualquier persona con el enlace y copiar el link.'}

## 5. Chatbot 2: embeddings con spaCy

Para el segundo chatbot usé embeddings preentrenados del modelo `es_core_news_md` de spaCy.

La diferencia principal es que, en vez de comparar solo palabras exactas o frecuentes, los embeddings buscan representar el significado del texto en un vector.  
Esto puede ayudar cuando la consulta usa palabras distintas a las del dataset, aunque no siempre funciona perfecto.

In [27]:
# Descargo el modelo con vectores usado para embeddings.
!python -m spacy download es_core_news_md -q

✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')


In [28]:
nlp = spacy.load("es_core_news_md")

print("Modelo cargado:", nlp.meta["name"])
print("Cantidad de dimensiones del vector:", nlp.vocab.vectors_length)

Modelo cargado: core_news_md
Cantidad de dimensiones del vector: 300


In [29]:
def vectorizar_con_spacy(texto):
    doc = nlp(normalizar_texto(texto))
    return doc.vector

embedding_matrix = np.vstack([
    vectorizar_con_spacy(pregunta)
    for pregunta in df_qa["pregunta"]
])

print("Tamaño de la matriz de embeddings:", embedding_matrix.shape)

Tamaño de la matriz de embeddings: (25, 300)


In [30]:
def responder_embeddings(consulta, umbral=0.55):
    consulta_vector = vectorizar_con_spacy(consulta).reshape(1, -1)

    similitudes = cosine_similarity(consulta_vector, embedding_matrix).flatten()
    indice_mejor = similitudes.argmax()
    mejor_similitud = similitudes[indice_mejor]

    if mejor_similitud < umbral:
        return {
            "modelo": "Embeddings spaCy",
            "consulta": consulta,
            "id_recuperado": None,
            "pregunta_recuperada": None,
            "similitud": round(float(mejor_similitud), 4),
            "respuesta": "No encontré una respuesta segura para esa consulta. Probá reformular la pregunta."
        }

    fila = df_qa.iloc[indice_mejor]
    return {
        "modelo": "Embeddings spaCy",
        "consulta": consulta,
        "id_recuperado": fila["id"],
        "pregunta_recuperada": fila["pregunta"],
        "similitud": round(float(mejor_similitud), 4),
        "respuesta": fila["respuesta"]
    }

responder_embeddings("¿Dónde miro cuándo tengo parcial?")

{'modelo': 'Embeddings spaCy',
 'consulta': '¿Dónde miro cuándo tengo parcial?',
 'id_recuperado': 'P04',
 'pregunta_recuperada': '¿Qué hago si no puedo acceder al campus?',
 'similitud': 0.6027,
 'respuesta': 'Si no podés acceder al campus, primero verificá tu usuario, contraseña y conexión. Si el problema sigue, tenés que comunicarte con soporte o administración.'}

## 6. Pruebas de funcionamiento

Para probar ambos chatbots preparé una lista de consultas.  
Algunas están escritas de forma parecida al dataset y otras están reformuladas, para ver si el chatbot puede recuperar la intención correcta aunque cambien algunas palabras.

In [31]:
consultas_prueba = [
    ("¿Cómo me anoto a una materia?", "P01"),
    ("No puedo entrar al campus, ¿qué hago?", "P04"),
    ("¿Dónde subo el trabajo práctico?", "P05"),
    ("¿Dónde miro cuándo tengo parcial?", "P07"),
    ("¿Hay recuperatorio si me va mal?", "P08"),
    ("¿Cómo contacto al profesor?", "P12"),
    ("¿Para qué usamos Colab?", "P14"),
    ("¿Cómo paso el link de mi notebook?", "P15"),
    ("¿Para qué sirve GitHub?", "P16"),
    ("¿Qué es un README?", "P17"),
    ("¿Qué diferencia hay entre tfidf y embeddings?", "P21"),
    ("¿Dónde solicito constancia de alumno regular?", "P24"),
    ("Necesito organizar mi estudio", "P25"),
    ("¿Cuál es el precio de la cuota?", None)
]

consultas_prueba

[('¿Cómo me anoto a una materia?', 'P01'),
 ('No puedo entrar al campus, ¿qué hago?', 'P04'),
 ('¿Dónde subo el trabajo práctico?', 'P05'),
 ('¿Dónde miro cuándo tengo parcial?', 'P07'),
 ('¿Hay recuperatorio si me va mal?', 'P08'),
 ('¿Cómo contacto al profesor?', 'P12'),
 ('¿Para qué usamos Colab?', 'P14'),
 ('¿Cómo paso el link de mi notebook?', 'P15'),
 ('¿Para qué sirve GitHub?', 'P16'),
 ('¿Qué es un README?', 'P17'),
 ('¿Qué diferencia hay entre tfidf y embeddings?', 'P21'),
 ('¿Dónde solicito constancia de alumno regular?', 'P24'),
 ('Necesito organizar mi estudio', 'P25'),
 ('¿Cuál es el precio de la cuota?', None)]

In [32]:
def probar_modelos(consultas):
    resultados = []

    for consulta, esperado in consultas:
        for funcion in [responder_tfidf, responder_embeddings]:
            resultado = funcion(consulta)
            resultado["id_esperado"] = esperado

            if esperado is None:
                resultado["correcto_estimado"] = resultado["id_recuperado"] is None
            else:
                resultado["correcto_estimado"] = resultado["id_recuperado"] == esperado

            resultados.append(resultado)

    return pd.DataFrame(resultados)

df_resultados = probar_modelos(consultas_prueba)
df_resultados[[
    "modelo",
    "consulta",
    "id_esperado",
    "id_recuperado",
    "pregunta_recuperada",
    "similitud",
    "correcto_estimado",
    "respuesta"
]]

,modelo,consulta,id_esperado,id_recuperado,pregunta_recuperada,similitud,correcto_estimado,respuesta
0,TF-IDF,¿Cómo me anoto a una materia?,P01,P01,¿Cómo me inscribo a una materia?,0.7434,True,Para inscribirte a una materia tenés que ingre...
1,Embeddings spaCy,¿Cómo me anoto a una materia?,P01,P01,¿Cómo me inscribo a una materia?,0.9739,True,Para inscribirte a una materia tenés que ingre...
2,TF-IDF,"No puedo entrar al campus, ¿qué hago?",P04,P04,¿Qué hago si no puedo acceder al campus?,0.7519,True,"Si no podés acceder al campus, primero verific..."
3,Embeddings spaCy,"No puedo entrar al campus, ¿qué hago?",P04,P04,¿Qué hago si no puedo acceder al campus?,0.9415,True,"Si no podés acceder al campus, primero verific..."
4,TF-IDF,¿Dónde subo el trabajo práctico?,P05,P06,¿Puedo entregar un trabajo práctico fuera de t...,0.3878,False,La entrega fuera de término depende de la polí...
5,Embeddings spaCy,¿Dónde subo el trabajo práctico?,P05,P19,¿Por qué tengo que comentar el código?,0.7353,False,Comentar el código ayuda a explicar qué hace c...
6,TF-IDF,¿Dónde miro cuándo tengo parcial?,P07,P08,¿Qué pasa si desapruebo un parcial?,0.2110,False,"Si desaprobás un parcial, generalmente podés a..."
7,Embeddings spaCy,¿Dónde miro cuándo tengo parcial?,P07,P04,¿Qué hago si no puedo acceder al campus?,0.6027,False,"Si no podés acceder al campus, primero verific..."
8,TF-IDF,¿Hay recuperatorio si me va mal?,P08,P18,¿Qué hago si Python me muestra un error?,0.3110,False,"Cuando Python muestra un error, conviene leer ..."
9,Embeddings spaCy,¿Hay recuperatorio si me va mal?,P08,P18,¿Qué hago si Python me muestra un error?,0.7523,False,"Cuando Python muestra un error, conviene leer ..."


In [33]:
resumen_modelos = (
    df_resultados
    .groupby("modelo")["correcto_estimado"]
    .agg(["sum", "count", "mean"])
    .rename(columns={"sum": "aciertos", "count": "total", "mean": "porcentaje_acierto"})
)

resumen_modelos["porcentaje_acierto"] = (resumen_modelos["porcentaje_acierto"] * 100).round(2)
resumen_modelos

,aciertos,total,porcentaje_acierto
modelo,,,
Embeddings spaCy,4,14,28.57
TF-IDF,7,14,50.00


In [34]:
# Casos donde alguno de los modelos no recuperó la pregunta esperada
df_resultados[df_resultados["correcto_estimado"] == False][[
    "modelo",
    "consulta",
    "id_esperado",
    "id_recuperado",
    "pregunta_recuperada",
    "similitud",
    "respuesta"
]]

,modelo,consulta,id_esperado,id_recuperado,pregunta_recuperada,similitud,respuesta
4,TF-IDF,¿Dónde subo el trabajo práctico?,P05,P06,¿Puedo entregar un trabajo práctico fuera de t...,0.3878,La entrega fuera de término depende de la polí...
5,Embeddings spaCy,¿Dónde subo el trabajo práctico?,P05,P19,¿Por qué tengo que comentar el código?,0.7353,Comentar el código ayuda a explicar qué hace c...
6,TF-IDF,¿Dónde miro cuándo tengo parcial?,P07,P08,¿Qué pasa si desapruebo un parcial?,0.2110,"Si desaprobás un parcial, generalmente podés a..."
7,Embeddings spaCy,¿Dónde miro cuándo tengo parcial?,P07,P04,¿Qué hago si no puedo acceder al campus?,0.6027,"Si no podés acceder al campus, primero verific..."
8,TF-IDF,¿Hay recuperatorio si me va mal?,P08,P18,¿Qué hago si Python me muestra un error?,0.3110,"Cuando Python muestra un error, conviene leer ..."
9,Embeddings spaCy,¿Hay recuperatorio si me va mal?,P08,P18,¿Qué hago si Python me muestra un error?,0.7523,"Cuando Python muestra un error, conviene leer ..."
10,TF-IDF,¿Cómo contacto al profesor?,P12,P03,¿Cómo entro al aula virtual?,0.3834,Para entrar al aula virtual tenés que usar el ...
11,Embeddings spaCy,¿Cómo contacto al profesor?,P12,P03,¿Cómo entro al aula virtual?,0.8639,Para entrar al aula virtual tenés que usar el ...
13,Embeddings spaCy,¿Para qué usamos Colab?,P14,P25,¿Cómo puedo organizarme mejor para estudiar?,0.8118,Una forma simple de organizarte es revisar el ...
17,Embeddings spaCy,¿Para qué sirve GitHub?,P16,P25,¿Cómo puedo organizarme mejor para estudiar?,0.7588,Una forma simple de organizarte es revisar el ...


## 7. Ejemplos individuales de uso

En estas celdas muestro algunas consultas puntuales para ver la respuesta completa de cada chatbot.

In [35]:
consulta = "¿Cómo comparto el enlace del colab?"

print("Respuesta TF-IDF")
print(responder_tfidf(consulta))

print("\nRespuesta con embeddings")
print(responder_embeddings(consulta))

Respuesta TF-IDF
{'modelo': 'TF-IDF', 'consulta': '¿Cómo comparto el enlace del colab?', 'id_recuperado': 'P15', 'pregunta_recuperada': '¿Cómo comparto un notebook de Google Colab?', 'similitud': 0.4318, 'respuesta': 'Para compartir un notebook de Colab tenés que usar el botón Compartir, cambiar los permisos a cualquier persona con el enlace y copiar el link.'}

Respuesta con embeddings
{'modelo': 'Embeddings spaCy', 'consulta': '¿Cómo comparto el enlace del colab?', 'id_recuperado': 'P15', 'pregunta_recuperada': '¿Cómo comparto un notebook de Google Colab?', 'similitud': 0.7361, 'respuesta': 'Para compartir un notebook de Colab tenés que usar el botón Compartir, cambiar los permisos a cualquier persona con el enlace y copiar el link.'}


In [36]:
consulta = "¿Qué pasa si desapruebo?"

print("Respuesta TF-IDF")
print(responder_tfidf(consulta))

print("\nRespuesta con embeddings")
print(responder_embeddings(consulta))

Respuesta TF-IDF
{'modelo': 'TF-IDF', 'consulta': '¿Qué pasa si desapruebo?', 'id_recuperado': 'P08', 'pregunta_recuperada': '¿Qué pasa si desapruebo un parcial?', 'similitud': 0.795, 'respuesta': 'Si desaprobás un parcial, generalmente podés acceder a una instancia de recuperatorio, aunque eso depende del reglamento de la materia.'}

Respuesta con embeddings
{'modelo': 'Embeddings spaCy', 'consulta': '¿Qué pasa si desapruebo?', 'id_recuperado': 'P08', 'pregunta_recuperada': '¿Qué pasa si desapruebo un parcial?', 'similitud': 0.8633, 'respuesta': 'Si desaprobás un parcial, generalmente podés acceder a una instancia de recuperatorio, aunque eso depende del reglamento de la materia.'}


In [37]:
consulta = "¿Dónde pido una constancia?"

print("Respuesta TF-IDF")
print(responder_tfidf(consulta))

print("\nRespuesta con embeddings")
print(responder_embeddings(consulta))

Respuesta TF-IDF
{'modelo': 'TF-IDF', 'consulta': '¿Dónde pido una constancia?', 'id_recuperado': 'P24', 'pregunta_recuperada': '¿Dónde pido un certificado de alumno regular?', 'similitud': 0.4237, 'respuesta': 'El certificado de alumno regular se solicita en administración o por el medio institucional indicado para trámites académicos.'}

Respuesta con embeddings
{'modelo': 'Embeddings spaCy', 'consulta': '¿Dónde pido una constancia?', 'id_recuperado': 'P23', 'pregunta_recuperada': '¿Qué significa regularizar una materia?', 'similitud': 0.7288, 'respuesta': 'Regularizar una materia significa cumplir las condiciones mínimas de cursada, como asistencia, trabajos prácticos y evaluaciones, según el reglamento.'}


## 8. Función opcional para probar el chatbot manualmente

La siguiente función permite escribir consultas de forma manual.  
No la dejo ejecutándose en bucle infinito por defecto para que el notebook sea más fácil de corregir.

In [38]:
def preguntar_chatbot(consulta, modelo="tfidf"):
    if modelo == "tfidf":
        return responder_tfidf(consulta)
    elif modelo == "embeddings":
        return responder_embeddings(consulta)
    else:
        return "Modelo no reconocido. Usar 'tfidf' o 'embeddings'."

preguntar_chatbot("¿Dónde veo los avisos de la materia?", modelo="tfidf")

{'modelo': 'TF-IDF',
 'consulta': '¿Dónde veo los avisos de la materia?',
 'id_recuperado': 'P13',
 'pregunta_recuperada': '¿Dónde se publican los avisos importantes?',
 'similitud': 0.3758,
 'respuesta': 'Los avisos importantes suelen publicarse en el aula virtual, por correo electrónico o en los canales oficiales de la institución.'}

In [39]:
preguntar_chatbot("¿Dónde veo los avisos de la materia?", modelo="embeddings")

{'modelo': 'Embeddings spaCy',
 'consulta': '¿Dónde veo los avisos de la materia?',
 'id_recuperado': 'P07',
 'pregunta_recuperada': '¿Dónde puedo ver las fechas de los parciales?',
 'similitud': 0.6721,
 'respuesta': 'Las fechas de parciales suelen publicarse en el aula virtual, en el cronograma de la materia o en los comunicados del docente.'}

## 9. Conclusiones

En este trabajo pude ver que un chatbot basado en recuperación de información depende mucho de la base de preguntas y respuestas que se le cargue.  
No alcanza con programar la función de búsqueda: también hay que pensar qué preguntas reales podría hacer una persona y escribir respuestas claras.

El modelo con TF-IDF funcionó bien cuando la consulta tenía palabras parecidas a las preguntas originales. Por ejemplo, si el usuario pregunta por "Colab", "GitHub", "parcial" o "trabajo práctico", el sistema suele recuperar una respuesta correcta porque esas palabras también aparecen en el dataset.

El chatbot con embeddings, en teoría, tiene la ventaja de poder encontrar relaciones aunque la pregunta esté escrita con otras palabras. En mis pruebas esto no siempre se cumplió. En algunos casos recuperó bien la respuesta, pero en otros se confundió con preguntas que tenían un significado general parecido y no con la intención exacta.

Una dificultad fue elegir los umbrales de similitud. Si el umbral es muy bajo, el chatbot responde aunque no esté seguro. Si el umbral es muy alto, puede rechazar preguntas que sí tenían una respuesta útil. Por eso agregué una respuesta alternativa cuando la similitud no supera cierto valor.

Las principales fallas aparecen cuando:

- la consulta es demasiado general.
- el usuario pregunta algo que no está en la base de conocimiento.
- dos preguntas del dataset son parecidas entre sí.
- aparecen sinónimos o formas de decir las cosas que el modelo no interpreta bien.

En comparación, TF-IDF me pareció más simple y fácil de explicar, pero depende mucho de las palabras exactas. En las pruebas de este notebook obtuvo mejores resultados que embeddings. Esto no significa que TF-IDF sea siempre superior, sino que para este dataset chico, estas preguntas de prueba y estos umbrales funcionó mejor. Los embeddings son más interesantes para capturar significado, aunque no son mágicos y también necesitan ajustes y pruebas.  
Como conclusión general, entendí que un chatbot de recuperación no "entiende" como una persona: busca la opción más parecida dentro de lo que ya conoce. Por eso, para mejorarlo, agregaría más preguntas reales, reformulaciones de una misma intención y una forma de registrar las preguntas que el chatbot no pudo responder.

## 10. Fuentes utilizadas

- Notebook de clase: `Chatbot_retrieval_.ipynb`.
- Notebook de clase: `Intro_Embeddings.ipynb`.
- Plantilla del trabajo práctico: `TP3_chatbot_retrieval_template_parte_1.ipynb`.
- Documentación oficial de Scikit-Learn: `TfidfVectorizer` y `cosine_similarity`.
- Documentación oficial de spaCy: modelos de lenguaje y vectores.
- Conversación con IA utilizada como apoyo para organizar la estructura del trabajo, revisar el enfoque y entender las diferencias entre TF-IDF y embeddings: https://chatgpt.com/share/6a0a25e7-31d4-83e9-974e-916b31693293.